In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/default/dsp_pipeline")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dsp_pipeline/advertisers.csv,advertisers.csv,401,1784447296000
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,events_day1.csv,3094332,1784447300000
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,events_day2.csv,4048441,1784447301000


## Bronze Layer – Raw Data Ingestion

Ingest raw vendor files as-is, with no cleaning or deduplication. 

Only metadata columns (`pipeline_ingest_time`, `source_file`) are added for lineage. 

v1/v2 schema drift is preserved by keeping the two event files as separate bronze tables — reconciliation happens explicitly in Silver, not silently here.

In [0]:
raw_path = "/Volumes/workspace/default/dsp_pipeline"
bronze_path = "/Volumes/workspace/default/dsp_pipeline/bronze"

print(raw_path)
print(bronze_path)

/Volumes/workspace/default/dsp_pipeline
/Volumes/workspace/default/dsp_pipeline/bronze


In [0]:
from pyspark.sql.functions import col, current_timestamp

def read_raw(filename):
    """Read a vendor CSV with header/schema inference, keeping source file path for lineage."""
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/{filename}")
        .select("*", "_metadata")
    )

events_day1_raw = read_raw("events_day1.csv")
events_day2_raw = read_raw("events_day2.csv")
advertisers_raw = read_raw("advertisers.csv")

### Schema drift: v1 (day1) vs v2 (day2)

- `spend` (v1) is renamed to `media_cost` (v2) — same meaning, different column name.

- `viewability` is new in v2, not present in v1.

- All other columns (`event_id`, `event_type`, `event_time`, `ingest_time`, 
  `advertiser_id`, `campaign_id`, `placement`, `device`, `geo`, `currency`) are unchanged.

Bronze keeps day1 and day2 as separate tables to preserve raw fidelity exactly as the 
vendor sent it. 

The rename/reconciliation (`media_cost` → `spend`, filling `viewability` 
as null for v1 rows) happens in Silver, where the unified schema is defined.

In [0]:
print("=== v1 (day1) schema ===")
events_day1_raw.printSchema()

print("=== v2 (day2) schema ===")
events_day2_raw.printSchema()

print("=== advertisers schema ===")
advertisers_raw.printSchema()

=== v1 (day1) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- advertiser_id: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- device: string (nullable = true)
 |-- geo: string (nullable = true)
 |-- spend: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- _metadata: struct (nullable = false)
 |    |-- file_path: string (nullable = false)
 |    |-- file_name: string (nullable = false)
 |    |-- file_size: long (nullable = false)
 |    |-- file_block_start: long (nullable = false)
 |    |-- file_block_length: long (nullable = false)
 |    |-- file_modification_time: timestamp (nullable = false)

=== v2 (day2) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time:

In [0]:
def add_bronze_metadata(df):
    """Attach ingest timestamp and source file path for auditability — no other changes."""
    return (
        df
        .withColumn("pipeline_ingest_time", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .drop("_metadata")
    )

bronze_events_day1 = add_bronze_metadata(events_day1_raw)
bronze_events_day2 = add_bronze_metadata(events_day2_raw)
bronze_advertisers = add_bronze_metadata(advertisers_raw)

In [0]:
bronze_events_day1.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day1")
bronze_events_day2.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day2")
bronze_advertisers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_advertisers")

In [0]:
# Sanity check only — row counts + samples, not part of the pipeline logic.
for t in ["bronze_events_day1", "bronze_events_day2", "bronze_advertisers"]:
    n = spark.table(f"workspace.default.{t}").count()
    print(f"{t}: {n} rows")

display(spark.table("workspace.default.bronze_events_day1").limit(5))
display(spark.table("workspace.default.bronze_events_day2").limit(5))

bronze_events_day1: 26118 rows
bronze_events_day2: 31498 rows
bronze_advertisers: 10 rows


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file
evt_7e681157,impression,2026-06-01T03:11:56Z,2026-06-01T09:17:54.000Z,adv_1151,cmp_1151_1,banner_300x250,mobile,IN,0.0194,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7d680fc4,impression,2026-06-01T06:32:35Z,2026-06-01T05:33:13.000Z,adv_1186,cmp_1186_2,interstitial,ctv,IN,0.0102,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_8068147d,click,2026-06-01T18:56:56Z,2026-06-01T01:16:09.000Z,adv_1151,cmp_1151_3,native_feed,mobile,GB,0.0212,INR,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7f6812ea,impression,2026-06-01T20:16:21Z,2026-06-01T03:29:28.000Z,adv_1172,cmp_1172_2,preroll,ctv,DE,0.018,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7a680b0b,impression,2026-06-01T07:43:12Z,2026-06-01T08:48:22.000Z,adv_1165,cmp_1165_1,banner_728x90,desktop,GB,0.0189,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,media_cost,currency,viewability,pipeline_ingest_time,source_file
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a715d374,impression,2026-06-04 20:03:11+05:30,2026-06-04T06:56:34.000Z,adv_1151,cmp_1151_1,interstitial,ctv,DE,0.0138,USD,0.48,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_aa15d82d,impression,2026-06-04 04:57:18+05:30,2026-06-04T19:06:57.000Z,adv_1172,cmp_1172_3,preroll,mobile,unknown,0.0131,USD,0.54,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a915d69a,impression,2026-06-04 10:59:30+05:30,2026-06-04T09:08:41.000Z,adv_1137,cmp_1137_2,banner_728x90,mobile,SG,0.0013,USD,0.87,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
